# 📗 EDA 기초 — 결합·집계로 데이터 요약

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

## 🎯 오늘의 목표
- [ ] `groupby` 로 데이터를 그룹으로 묶어 요약한다.
- [ ] 여러 키·여러 집계(`agg`)를 한 번에 계산한다.
- [ ] `pivot_table` 로 행·열 교차 요약표를 만든다.
- [ ] `crosstab` 으로 빈도표(교차표)를 만든다.
- [ ] `merge` 로 두 표를 공통 키로 이어 붙인다.
- [ ] `concat` 으로 표를 위·아래(또는 옆)로 이어 붙인다.
- [ ] 요약 결과에서 **인사이트**를 뽑아낸다.

## ⏪ 복습 — 지난 단원: pandas 기초
지난 단원에서는 표 한 장을 다루는 기본기를 익혔습니다.
- `pd.read_csv` 로 파일을 읽고, `head`·`info`·`describe` 로 훑어봤죠.
- 대괄호·`loc`·`iloc` 로 원하는 열·행을 골라내고, **불리언 필터**로 조건에 맞는 행만 추렸습니다.
- 결측치(`fillna`)와 이상치를 정제해 데이터를 깨끗하게 만들었습니다.

이제 깨끗한 표 한 장을 넘어, **"요일별 평균은?", "성별·흡연 여부로 나눠 보면?"** 같은 **그룹 단위 질문**에 답할 차례입니다. 그리고 흩어진 두 표를 **하나로 잇는 법**(merge·concat)도 배웁니다. 오늘은 레스토랑 팁 데이터로 **요약의 기술**을 익힙니다.

## 오늘의 데이터 — 레스토랑 팁 기록
`restaurant.csv` 는 어느 식당의 영수증 244건입니다. 손님이 낸 **팁(tip)** 이 요일·시간대·인원수에 따라 어떻게 달라지는지 살펴봅니다.

- `total_bill` — 전체 식사 금액(달러)
- `tip` — 팁 금액(달러)
- `sex` — 결제자 성별 (Male / Female)
- `smoker` — 흡연석 여부 (Yes / No)
- `day` — 요일 (Thur / Fri / Sat / Sun)
- `time` — 시간대 (Lunch / Dinner)
- `size` — 함께 온 인원수

In [ ]:
# [제공 코드] 오늘 내내 쓸 pandas 를 불러옵니다.
import pandas as pd

In [ ]:
# 데이터를 불러와 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv("data/restaurant.csv")
print("행·열 크기:", df.shape)   # (244, 7) — 244행 7열
print("\n[앞 5행] head()")
display(df.head())
print("\n[열·자료형·결측] info()")
df.info()
print("\n[수치 요약] describe()")
display(df.describe())

---
# 1. groupby — 그룹으로 묶어 요약하기

## 왜 필요할까요?
"전체 팁 평균"은 `df['tip'].mean()` 한 줄이면 됩니다. 하지만 실무 질문은 대부분 **"~별로"** 입니다 — "**요일별** 평균 팁은?", "**인원수별** 평균 식사 금액은?".

이럴 때 `groupby` 를 씁니다. **① 같은 값끼리 그룹으로 묶고 → ② 각 그룹을 하나의 숫자로 요약**하는 2단계 도구예요.

> **일상 비유** — 영수증 244장을 요일이 적힌 4개의 바구니(Thur·Fri·Sat·Sun)에 나눠 담고, 바구니마다 팁의 평균을 구하는 것. 결과는 **요일 하나당 한 줄**이 됩니다.

| 문법 | 하는 일 |
|---|---|
| `df.groupby('day')['tip'].mean()` | 요일별 팁 **평균** |
| `df.groupby('day')['tip'].sum()` | 요일별 팁 **합계** |
| `df.groupby('day').size()` | 요일별 **행 개수**(그룹 크기) |
| `df.groupby('day')['tip'].count()` | 요일별 결측 아닌 값 개수 |

> `groupby('day')` 의 결과 인덱스는 **요일**이 됩니다. 244행이 요일 4줄로 줄어들죠 — 이게 "요약"입니다.

<img src="images/groupby_split_apply_combine.png" alt="groupby 3단계 — 나누고 계산하고 합치기" width="900"/>

In [ ]:
# 요일별 팁 평균 — groupby 로 244행을 요일 4줄로 요약
day_tip = df.groupby('day')['tip'].mean()
print(day_tip.round(3))

# 요약한 결과도 결국 표(Series)라 정렬할 수 있습니다 — 높은 순으로 줄 세우면 순위가 한눈에.
print('\n--- 평균 팁 높은 순 ---')
print(day_tip.sort_values(ascending=False).round(3))

In [ ]:
# 같은 방식으로 합계·그룹 크기도 구할 수 있습니다.
print("요일별 팁 합계:")
print(df.groupby('day')['tip'].sum())
print("\n요일별 주문 건수(size):")
print(df.groupby('day').size())

In [ ]:
# 인원수(size 열)별 평균 식사 금액 — 숫자 열로도 그룹을 만들 수 있어요.
# 주의: 'size' 는 인원수 '열 이름'입니다. groupby(...).size() 의 그룹 크기 메서드와 이름만 같아요.
# 인원이 많을수록 식사 금액이 커지는 경향이 보입니다.
df.groupby('size')['total_bill'].mean()

### 🖐️ 함께 따라하기 — 프로그램별 평균 운동시간
데모는 **레스토랑 팁** 데이터였죠. 따라하기는 **다른 도메인 — 피트니스 센터 이용 기록**(`gym_visits.csv`)으로 같은 기술을 연습합니다. 먼저 데이터를 불러와 훑어본 뒤 그룹 요약을 해 봅시다.

> ⚠️ 이 셀에서 만드는 `gym` 을 **이후 따라하기에서 계속 씁니다** — 건너뛰지 말고 꼭 실행하고 넘어가세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '레스토랑 팁'이었죠. 이번엔 다른 데이터(피트니스 센터 이용 기록)로 연습합니다.
# 1) pd.read_csv 로 data/gym_visits.csv 를 읽어 변수 gym 에 담는다
# 2) head() 로 앞부분을, info() 로 열·자료형을 먼저 살펴본다
# 3) gym 을 '프로그램' 으로 그룹화해 '운동시간' 의 평균을 구한다
# 4) 결과를 출력해 평균 운동시간이 가장 긴 프로그램을 확인한다

### ✅ 바로 확인 퀴즈
**1.** `df.groupby('day')['tip'].mean()` 의 결과는 왜 요일마다 **한 줄**(총 4줄)일까요?

<details><summary>정답 보기</summary>

groupby 가 같은 요일끼리 그룹으로 묶어 **각 그룹을 평균값 하나로 요약**하기 때문입니다. 244행이 요일 4그룹으로 줄어듭니다.

</details>

**2.** 위 데이터에서 평균 팁이 **가장 높은 요일**은 어디일까요?

<details><summary>정답 보기</summary>

**Sun(일요일)**, 약 3.255달러입니다. (Fri 2.735 < Thur 2.771 < Sat 2.993 < Sun 3.255)

</details>

**3.** 요일별 **주문 건수**를 구하려면 어떤 메서드를 쓸까요?

<details><summary>정답 보기</summary>

`df.groupby('day').size()` 입니다. size 는 결측 포함 그룹 전체 행 수, 특정 열의 `.count()` 는 그 열의 결측을 뺀 개수예요.

</details>

---
# 2. groupby 심화 — 여러 키·여러 집계

## 왜 필요할까요?
그룹 기준이 하나로 부족할 때가 있습니다 — "**요일 × 시간대**별 평균은?". 그리고 한 그룹을 여러 각도로 보고 싶을 때도 있죠 — "평균**과** 합계**와** 개수를 한 번에".

- **여러 키로 묶기**: `groupby(['day', 'time'])` — 리스트로 여러 열을 넘기면 조합별로 그룹이 만들어집니다.
- **여러 집계 한 번에**: `.agg(['mean', 'sum', 'count'])` — 집계 함수 이름을 리스트로 넘깁니다.
- **그룹 키를 열로 두기**: `as_index=False` — 그룹 키를 인덱스가 아니라 **일반 열**로 돌려받아 이후 다루기 편하게 합니다.

| 문법 | 하는 일 |
|---|---|
| `df.groupby(['day','time'])['total_bill'].mean()` | 요일×시간대별 평균 |
| `df.groupby('day')['tip'].agg(['mean','sum','count'])` | 한 그룹에 여러 집계 동시에 |
| `df.groupby('day', as_index=False)['tip'].mean()` | 그룹 키(day)를 일반 열로 |

In [ ]:
# 여러 키 — 요일 × 시간대 조합별 평균 식사 금액
df.groupby(['day', 'time'])['total_bill'].mean()

In [ ]:
# 여러 집계 — 요일별 팁의 평균·합계·개수를 한 표로 (리스트 형태)
display(df.groupby('day')['tip'].agg(['mean', 'sum', 'count']).round(3))

# 열마다 다른 집계가 필요할 땐 딕셔너리로 — {'열이름': '집계함수'}
# 실무에서 가장 많이 쓰는 형태입니다: 금액은 평균, 팁은 합계, 인원은 최댓값처럼.
display(df.groupby('day').agg({
    'total_bill': 'mean',    # 식사 금액은 평균
    'tip': 'sum',            # 팁은 합계
    'size': 'max',           # 인원수는 최댓값
}).round(3))

In [ ]:
# as_index=False — 그룹 키(day)가 인덱스가 아니라 일반 열로 나옵니다.
df.groupby('day', as_index=False)['tip'].mean()

### 🖐️ 함께 따라하기 — 프로그램 × 회원등급
키를 **두 개**로 묶어 봅시다. 프로그램별로, 또 그 안에서 회원등급별로 나눠 평균 소모칼로리를 구합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym 을 '프로그램'·'회원등급' 두 열로 그룹화한다 (리스트로 묶어 전달)
# 2) 그 그룹의 '소모칼로리' 평균을 구한다
# 3) 결과를 출력한다 (프로그램 4종 × 등급 2종 = 8줄이 나오는지 확인)

### ✅ 바로 확인 퀴즈
**1.** `df.groupby(['day','time'])` 처럼 키를 **리스트**로 주면 결과는 어떻게 될까요?

<details><summary>정답 보기</summary>

요일과 시간대의 **조합**마다 그룹이 생깁니다. 결과 인덱스가 (day, time) 2단계가 되죠.

</details>

**2.** 요일별 팁의 **평균·합계·개수**를 한 번에 구하는 코드는?

<details><summary>정답 보기</summary>

`df.groupby('day')['tip'].agg(['mean','sum','count'])` — 집계 함수 이름을 리스트로 넘깁니다.

</details>

---
# 3. pivot_table — 행·열 교차 요약표

## 왜 필요할까요?
`groupby(['day','time'])` 의 결과는 세로로 길게 늘어져 한눈에 안 들어옵니다. 같은 요약을 **행=요일, 열=시간대** 격자(엑셀 피벗테이블처럼)로 펼치면 훨씬 읽기 좋습니다. 그게 `pivot_table` 이에요.

| 인자 | 뜻 |
|---|---|
| `index='day'` | 표의 **행**이 될 기준 |
| `columns='time'` | 표의 **열**이 될 기준 |
| `values='total_bill'` | 셀에 채울 **값** |
| `aggfunc='mean'` | 값을 요약할 방법(평균·합계 등) |
| `margins=True` | 행·열 **총계(All)** 를 덧붙임 |

- `pivot` vs `pivot_table`: `pivot` 은 재배치만 하고(같은 조합이 중복이면 에러), `pivot_table` 은 중복을 **집계(aggfunc)** 해 주므로 요약표엔 `pivot_table` 을 씁니다.

<img src="images/pivot_long_to_wide.png" alt="pivot_table — 긴 표를 행×열 격자로" width="900"/>

In [ ]:
# 요일 × 시간대 평균 식사 금액을 격자표로
df.pivot_table(index='day', columns='time', values='total_bill', aggfunc='mean')

In [ ]:
# 결과에 NaN(빈 칸)이 보이나요?
# Sat·Sun 의 Lunch 칸이 NaN 입니다 — 이 데이터에 '주말 점심' 주문이 한 건도 없어서예요.
# NaN 은 "값이 틀렸다"가 아니라 "해당 조합이 아예 없다"는 뜻입니다.
# margins=True 로 행·열 총계(All)도 함께 봅시다.
display(df.pivot_table(index='day', columns='time', values='total_bill',
                       aggfunc='mean', margins=True).round(2))

# fill_value= 로 빈 칸을 원하는 값으로 채울 수 있습니다 (여기선 0).
# ⚠️ 단, '주문이 없다'와 '금액이 0원이다'는 다른 뜻이니 0으로 채울지는 신중히 판단하세요.
#    개수(count)를 셀 때는 0이 자연스럽지만, 평균 금액을 0으로 채우면 해석이 왜곡됩니다.
display(df.pivot_table(index='day', columns='time', values='total_bill',
                       aggfunc='count', fill_value=0))

### 🖐️ 함께 따라하기 — 요일 × 시간대 격자표
`pivot_table` 로 **요일(행) × 시간대(열)** 평균 운동시간 격자표를 만들어 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym 의 pivot_table 을 쓴다 — 행(index)은 '요일', 열(columns)은 '시간대'
# 2) 값(values)은 '운동시간', 집계(aggfunc)는 평균으로 지정한다
# 3) 결과를 소수 1자리로 반올림해 출력한다 (7행 × 3열 격자가 나오는지 확인)

### ✅ 바로 확인 퀴즈
**1.** pivot_table 결과에서 Sat·Sun 의 Lunch 칸이 NaN 인 이유는?

<details><summary>정답 보기</summary>

이 데이터에 **주말 점심 주문이 없기 때문**입니다. 그 조합에 해당하는 행이 0건이라 집계할 값이 없어 NaN 이 됩니다.

</details>

**2.** 행·열의 **총계(All)** 까지 표에 넣으려면 어떤 인자를 줄까요?

<details><summary>정답 보기</summary>

`margins=True` 를 줍니다.

</details>

---
# 4. crosstab — 빈도표(교차표)

## 왜 필요할까요?
"요일별로 Lunch·Dinner 주문이 **각각 몇 건**인가?" 처럼 **개수를 세는** 교차표가 자주 필요합니다. `pd.crosstab` 은 이 **빈도표**에 특화된 도구예요.

- `pd.crosstab(df['day'], df['time'])` — 요일(행) × 시간대(열)별 **건수**.
- `normalize='index'` — 각 행을 **비율**(행 합이 1)로. 행 안에서의 구성비를 볼 때 유용.
- `crosstab` vs `pivot_table`: 둘 다 교차표지만, **crosstab 은 빈도(개수) 세기에 특화**되어 값(values) 인자가 필요 없습니다. 값을 집계(평균 등)하려면 pivot_table 을 씁니다.

In [ ]:
# 요일 × 시간대 주문 건수 교차표
pd.crosstab(df['day'], df['time'])

In [ ]:
# normalize='index' — 각 요일 안에서 Lunch/Dinner 비율(행 합이 1)
# Thur 는 대부분 Lunch, Sat·Sun 은 전부 Dinner 임이 한눈에 보입니다.
display(pd.crosstab(df['day'], df['time'], normalize='index').round(3))

# normalize='columns' — 방향을 바꿔서, 각 시간대 안에서 요일 비율(열 합이 1)
# "점심 손님은 주로 무슨 요일에 오나?"처럼 질문이 달라지면 정규화 방향도 달라집니다.
display(pd.crosstab(df['day'], df['time'], normalize='columns').round(3))

### 🖐️ 함께 따라하기 — 프로그램 × 회원등급 빈도표
평균이 아니라 **몇 건인지**가 궁금할 때는 `crosstab` 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pd.crosstab 으로 행에 gym['프로그램'], 열에 gym['회원등급'] 을 놓아 방문 건수 교차표를 만든다
# 2) 결과를 출력한다
# 3) 이어서 normalize='index' 를 주면 각 프로그램 안에서의 등급 '비율'(행 합=1)이 나온다 — 그것도 출력해 본다

### ✅ 바로 확인 퀴즈
**1.** `pd.crosstab` 과 `pivot_table` 의 가장 큰 차이는?

<details><summary>정답 보기</summary>

crosstab 은 **빈도(개수) 세기**에 특화되어 값 인자 없이 조합별 건수를 셉니다. 값을 집계(평균 등)하려면 pivot_table 을 씁니다.

</details>

**2.** 각 행을 비율(행 합=1)로 바꾸려면 어떤 인자를 줄까요?

<details><summary>정답 보기</summary>

`normalize='index'` 인자를 줍니다. (열 기준은 `normalize='columns'`)

</details>

---
# 5. merge — 두 표를 키로 이어 붙이기

## 왜 필요할까요?
분석에 필요한 정보가 **여러 표에 흩어져** 있을 때가 많습니다. 예를 들어 팁 데이터엔 요일만 있고, "그 요일이 주말인가?"는 **다른 표**(`day_info.csv`)에 있죠. 공통 열(**키**, 여기선 `day`)을 기준으로 두 표를 옆으로 합치는 게 `merge` 입니다.

`how=` 로 **어느 쪽 키를 남길지** 정합니다.
- `how='inner'` (기본): **양쪽 모두에 있는** 키만 남깁니다(교집합).
- `how='left'`: **왼쪽 표의 키는 전부** 남기고, 오른쪽에 짝이 없으면 NaN.
- `how='right'`: **오른쪽 표의 키**를 전부 남깁니다.
- `how='outer'`: **양쪽의 모든 키**를 남깁니다(합집합).

<img src="images/merge_how_4.png" alt="merge 결합 방식 4종" width="820"/>

In [ ]:
# 병합할 작은 표를 읽어옵니다. 요일마다 '주말인지(is_weekend)' 정보가 있어요.
day_info = pd.read_csv("data/day_info.csv")
day_info

In [ ]:
# 'day' 를 키로 두 표를 병합 — 팁 데이터에 is_weekend·sort_order 열이 붙습니다.
merged = df.merge(day_info, on='day')
print("병합 결과 크기:", merged.shape)   # (244, 9) — 244행 유지, 열이 7->9로 늘어남
merged.head(3)

### `how=` — 짝이 없는 행을 어떻게 할까

위 예시는 `day_info` 에 4개 요일이 **모두** 있어서 행 수가 그대로였습니다. 하지만 실무의 참조표는 **비어 있는 키**가 흔합니다. 그때 `how=` 가 결과를 가릅니다.

| `how` | 남기는 행 | 짝이 없으면 |
|---|---|---|
| `'inner'`(기본) | 양쪽에 **다 있는** 키만 | 그 행이 **사라짐** |
| `'left'` | **왼쪽 표 전부** | 오른쪽 열이 `NaN` |
| `'right'` | **오른쪽 표 전부** | 왼쪽 열이 `NaN` |
| `'outer'` | **양쪽 전부** | 없는 쪽이 `NaN` |

> ⚠️ **행이 늘어나는 함정** — 행 수가 유지되려면 **오른쪽 표의 키가 중복 없이 하나씩**이어야 합니다. 오른쪽에 같은 키가 2번 있으면 왼쪽 행이 **2배로 불어납니다**(merge 사고 1순위).

In [ ]:
# how= 차이를 눈으로 — 일부러 '금요일 정보가 빠진' 참조표를 만들어 비교합니다.
day_info_partial = day_info[day_info['day'] != 'Fri']   # Fri 행이 없는 표
print("참조표에 있는 요일:", list(day_info_partial['day']))

inner_df = df.merge(day_info_partial, on='day', how='inner')   # 짝 있는 것만
left_df  = df.merge(day_info_partial, on='day', how='left')    # 왼쪽(주문)은 전부 유지

print("inner:", inner_df.shape, "-> 금요일 주문이 통째로 빠졌습니다")
print("left :", left_df.shape,  "-> 행 수는 유지, 대신 금요일은 정보가 NaN")
print("left 의 is_weekend 결측 개수:", left_df['is_weekend'].isna().sum(), "(= 금요일 주문 수)")

print("\n오른쪽 표 키가 중복 없이 하나씩인가:", day_info['day'].is_unique)

### 키 이름이 다르거나, 열 이름이 겹칠 때

실무에서 두 표를 붙일 때 걸리는 두 가지가 더 있습니다.

- **키 이름이 서로 다를 때** — 왼쪽은 `day`, 오른쪽은 `요일` 처럼 같은 뜻인데 이름이 다르면 `on=` 을 못 씁니다. 이때는 `left_on='day', right_on='요일'` 로 양쪽 키를 따로 지정합니다.
- **키 말고 다른 열 이름이 겹칠 때** — 두 표에 똑같이 `note` 열이 있으면 pandas 가 자동으로 `note_x`(왼쪽)·`note_y`(오른쪽)로 이름을 바꿉니다. 이 꼬리표를 `suffixes=('_주문', '_요일정보')` 처럼 **알아보기 쉽게** 지정할 수 있습니다.

In [ ]:
# 키 이름이 다른 경우 — left_on / right_on
day_info_kr = day_info.rename(columns={'day': '요일'})     # 오른쪽 표의 키 이름을 '요일'로 바꿔 둠
print("왼쪽 키: 'day' / 오른쪽 키: '요일'")
merged_kr = df.merge(day_info_kr, left_on='day', right_on='요일')
print("병합 성공:", merged_kr.shape, "— 키가 양쪽 다 남습니다(day·요일)")

# 열 이름이 겹치는 경우 — suffixes
left = df[['day', 'total_bill']].head(3).copy()
right = day_info.copy()
left['note'] = '주문기록'        # 양쪽에 똑같은 이름의 열을 일부러 만들어 봅니다
right['note'] = '요일정보'

print("\n--- suffixes 를 안 주면 자동으로 _x / _y ---")
display(left.merge(right, on='day'))

print("--- suffixes 로 알아보기 쉽게 ---")
display(left.merge(right, on='day', suffixes=('_주문', '_요일정보')))

In [ ]:
# 병합 덕분에 '주말 여부'로 그룹 요약이 가능해졌습니다.
# 주말(True) 평균 팁이 평일(False)보다 높네요.
merged.groupby('is_weekend')['tip'].mean()

### 🖐️ 함께 따라하기 — 프로그램 정보 붙여 카테고리별 요약
`program_info.csv` 에는 프로그램마다 **카테고리·난이도·정원**이 들어 있습니다. 이 표를 붙이면 원래 없던 기준(카테고리)으로 요약할 수 있어요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pd.read_csv 로 data/program_info.csv 를 읽어 변수 program_info 에 담고 내용을 확인한다
# 2) gym 과 program_info 를 '프로그램' 키로 merge 해 변수 gym_full 에 담는다
# 3) 병합 뒤 열이 늘었는지 shape 로 확인한다
# 4) gym_full 을 '카테고리' 로 그룹화해 '소모칼로리' 평균을 구해 출력한다

### ✅ 바로 확인 퀴즈
**1.** `how='inner'` 와 `how='left'` 의 차이는?

<details><summary>정답 보기</summary>

inner 는 **양쪽에 다 있는 키**만 남깁니다(교집합). left 는 **왼쪽 표의 키를 전부** 남기고 오른쪽에 짝이 없으면 NaN 을 채웁니다.

</details>

**2.** `df.merge(day_info, on='day')` 의 결과 행 수가 244로 그대로인 이유는?

<details><summary>정답 보기</summary>

day_info 에 4개 요일이 **모두** 있어서 팁 데이터의 모든 행이 짝을 찾기 때문입니다. 빠지는 행이 없어 244행이 유지됩니다.

</details>

---
# 6. concat — 표를 이어 붙이기

## 왜 필요할까요?
merge 가 **키를 맞춰 옆으로** 합치는 것이라면, `concat` 은 **위·아래(또는 옆)로 그냥 이어 붙이는** 것입니다. "1월 데이터 + 2월 데이터"처럼 **같은 구조의 표 여러 개를 쌓을** 때 씁니다.

- `pd.concat([df1, df2])` — 기본은 **행 방향**(세로로 쌓기).
- `ignore_index=True` — 이어 붙인 뒤 인덱스를 0부터 새로 매김(원래 인덱스가 중복되지 않게).
- `axis=1` — **열 방향**(옆으로 붙이기).
- merge 와 차이: merge 는 **공통 키**로 맞춰 합치고, concat 은 **키 없이 위치·이름 그대로** 이어 붙입니다.

In [ ]:
# 표의 앞 50행과 뒤 50행을 세로로 이어 붙이기
head_part = df.head(50)
tail_part = df.tail(50)
stacked = pd.concat([head_part, tail_part], ignore_index=True)
print("50 + 50 =", len(stacked), "행")   # 100
stacked.shape

`axis=1` 을 주면 **열 방향(옆으로)** 이어 붙입니다 — 행 위치(인덱스)를 그대로 맞대어 열을 늘리는 것이라, `merge` 처럼 **공통 키로 짝을 찾지 않습니다**. 키를 맞춰 합치려면 merge, 위치 그대로 나란히 붙이려면 `concat(axis=1)` 을 씁니다.

In [ ]:
# axis=1 — 열 방향으로 옆에 이어 붙이기 (같은 행 위치끼리 맞댄다)
left_cols = df[['total_bill', 'tip']].head(3)    # 앞 3행의 금액·팁 두 열
right_cols = df[['day', 'time']].head(3)         # 같은 앞 3행의 요일·시간대 두 열
side_by_side = pd.concat([left_cols, right_cols], axis=1)   # 옆으로 붙여 4열이 됨
print("열 방향 결합 크기:", side_by_side.shape)   # (3, 4)
side_by_side

### 🖐️ 함께 따라하기 — 두 조각 이어 붙이기
표의 앞뒤 조각을 잘라 **세로로** 이어 붙여 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym 의 앞 30행(head)과 뒤 20행(tail)을 각각 변수에 담는다
# 2) pd.concat 으로 두 조각을 세로로 이어 붙인다 (인덱스는 0부터 다시 매기기)
# 3) 이어 붙인 결과의 행 수가 50인지 len 으로 확인한다

### ✅ 바로 확인 퀴즈
**1.** merge 와 concat 의 차이를 한 문장으로?

<details><summary>정답 보기</summary>

merge 는 **공통 키를 맞춰 옆으로** 합치고, concat 은 **키 없이 위·아래(또는 옆)로 그냥 이어 붙입니다.**

</details>

**2.** `pd.concat` 에서 `axis=0` 과 `axis=1` 은 결과 표의 모양이 어떻게 달라질까요?

<details><summary>정답 보기</summary>

`axis=0`(기본)은 **행 방향(세로)** 으로 쌓아 **행 수가 늘어나고**, `axis=1`은 **열 방향(가로)** 으로 붙여 **열 수가 늘어납니다**. 세로로 쌓을 땐 열 이름을, 가로로 붙일 땐 행 인덱스를 기준으로 정렬합니다.

</details>

---
## 🚀 응용 클론코딩 — 집계로 짧은 리포트 만들기

오늘 배운 것을 **한 흐름**으로 이어 봅시다: 병합 → 여러 키 그룹 → 여러 집계 → 격자표 → 정렬.

**미션**: 피트니스 데이터에 프로그램 정보를 붙여, **카테고리 × 회원등급별 이용 요약표**를 만들고 "가장 오래 운동하는 조합"을 찾아냅니다.

> 이렇게 만든 요약표 한 장이 곧 리포트의 근거가 됩니다. 다음 시간에는 이 표를 **그림**으로 바꿉니다.

In [ ]:
# 🖐️ 함께 따라하기 — 집계 리포트 (아래 순서대로 직접 작성해 보세요)
# 1) gym 과 program_info 를 '프로그램' 키로 merge 해 gym_full 에 담는다
# 2) gym_full 을 ['카테고리', '회원등급'] 두 키로 그룹화하고,
#    딕셔너리 형태의 agg 로 운동시간은 평균, 소모칼로리는 평균, 방문 건수는 크기(size)를 구한다
#    (힌트: agg({'운동시간': 'mean', '소모칼로리': 'mean'}) 뒤에 건수는 따로 세도 된다)
# 3) 결과를 소수 1자리로 반올림해 출력한다
# 4) pivot_table 로 행='카테고리', 열='회원등급', 값='운동시간' 평균 격자표를 만들어 출력한다
# 5) 2)의 결과를 '운동시간' 내림차순으로 정렬해 가장 오래 운동하는 조합을 확인한다

---
## 오늘의 인사이트 — 요약이 말해 준 것

집계 몇 줄로 이 식당에 대해 이런 사실들을 알아냈습니다.
- **주말(Sat·Sun)의 평균 팁이 평일보다 높다** — 주말 3.115달러 vs 평일 2.763달러(§5 병합 결과). 식사 금액도 같은 방향인지는 `merged.groupby('is_weekend')['total_bill'].mean()` 으로 직접 확인해 보세요.
- **팁이 가장 높은 요일은 일요일(Sun, 3.255달러)**, 가장 낮은 요일은 금요일(Fri)입니다.
- **인원수가 많을수록 식사 금액이 커진다** — size 1명 7.2달러 → 4명 28.6달러.
- **요일마다 영업 패턴이 다르다** — Thur 는 대부분 점심, Sat·Sun 은 전부 저녁 장사(crosstab).

숫자를 그룹으로 묶어 요약하니, 흩어진 244장의 영수증에서 **패턴**이 드러났습니다. 이것이 EDA(탐색적 데이터 분석)의 첫걸음입니다.

## 이번 강의 정리

| 하고 싶은 일 | 함수 | 쓰임 |
|---|---|---|
| 그룹별 요약 | `df.groupby('키')['열'].mean()` | ~별 평균·합계·개수 |
| 여러 키·여러 집계 | `groupby([...]).agg([...])` | 조합별·여러 통계 한 번에 |
| 행·열 교차 요약표 | `df.pivot_table(index, columns, values, aggfunc)` | 격자 요약(엑셀 피벗) |
| 빈도 교차표 | `pd.crosstab(행, 열)` | 조합별 건수 세기 |
| 두 표를 키로 합치기 | `df.merge(other, on='키', how=...)` | 흩어진 정보 옆으로 결합 |
| 표를 이어 붙이기 | `pd.concat([df1, df2])` | 같은 구조 표 위·아래로 쌓기 |

## ⏭️ 예고 — 다음: 이 요약을 그림으로 (시각화)
오늘은 숫자 표로 요약했습니다. 하지만 "주말이 팁이 높다"는 사실은 **막대그래프 하나**로 훨씬 강렬하게 보여줄 수 있죠.

다음 시간에는 **matplotlib·seaborn** 으로 오늘의 요약을 **그림**으로 바꿉니다 — 분포를 보는 히스토그램·박스플롯, 그룹을 비교하는 막대그래프, 두 값의 관계를 보는 산점도까지. "표로 요약 → 그림으로 전달"이 EDA 의 완성입니다.